# Card, mobile wallet or cash: payment choice by age

**Multinomial Logistic Regression** · a worked example from the StatNex teaching guide

**The story.** A shop asked 59 customers how they prefer to pay: by card, with a mobile wallet on their phone, or in cash. It also knows each customer's age. The owner is wondering whether to stop taking cash.

**The question.** How does age shift the choice between card, mobile wallet and cash?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/multinomial_logistic_regression.csv` (59 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Age,Preferred payment
1.0,42.0,Card
2.0,25.0,Card
3.0,24.0,Mobile wallet
4.0,44.0,Card
5.0,32.0,Card
6.0,23.0,Mobile wallet
7.0,44.0,Cash
8.0,32.0,Mobile wallet
9.0,59.0,Mobile wallet
10.0,63.0,Cash
11.0,26.0,Card
12.0,49.0,Cash
13.0,47.0,Card
14.0,43.0,Card
15.0,46.0,Card
16.0,50.0,Card
17.0,47.0,Card
18.0,20.0,Mobile wallet
19.0,50.0,Card
20.0,39.0,Mobile wallet
21.0,39.0,Cash
22.0,40.0,Cash
23.0,63.0,Card
24.0,62.0,Cash
25.0,26.0,Mobile wallet
26.0,62.0,Cash
27.0,19.0,Card
28.0,40.0,Mobile wallet
29.0,39.0,Card
30.0,43.0,Cash
31.0,27.0,Mobile wallet
32.0,26.0,Mobile wallet
33.0,31.0,Mobile wallet
34.0,47.0,Card
35.0,54.0,Card
36.0,68.0,Card
37.0,65.0,Cash
38.0,22.0,Mobile wallet
39.0,70.0,Card
40.0,63.0,Card
41.0,18.0,Mobile wallet
42.0,29.0,Card
43.0,37.0,Card
44.0,40.0,Card
45.0,39.0,Mobile wallet
46.0,66.0,Cash
47.0,38.0,Mobile wallet
48.0,58.0,Cash
49.0,27.0,Cash
50.0,56.0,Card
51.0,46.0,Card
52.0,25.0,Card
53.0,51.0,Card
54.0,18.0,Card
55.0,46.0,Card
56.0,28.0,Card
57.0,23.0,Mobile wallet
58.0,49.0,Mobile wallet
59.0,34.0,Mobile wallet

These cells save `categories.py`, `collinear_columns.py`: steps that no library does, written out. The analysis imports them.

In [ ]:
%%writefile categories.py
"""The categories of a qualitative variable, in the order StatNex lists them.

Each value is written as the sheet shows it (a code read as 2.0 is "2"). When every
category is a number they go from the smallest up: 1, 2, 3, 10, where sorting them as
text would put 10 between 1 and 2. Otherwise they are sorted by their bytes in
Shift_JIS (CP932) -- the order a Japanese Excel sorts by, which for kanji runs by
reading. For plain English labels this is ordinary alphabetical order with capitals
first. pandas has no such sort, so it is written out here.
"""

import numpy as np


def as_text(value):
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)


def cp932_key(text):
    try:
        return text.encode("cp932")
    except UnicodeEncodeError:
        return text.encode("utf-8")


def category_order(labels):
    """The distinct labels in order: numbers from the smallest up, else by CP932 bytes."""
    labels = set(labels)
    try:
        return sorted(labels, key=float)
    except ValueError:
        return sorted(labels, key=cp932_key)


def categories(column):
    """(the categories in order, each case's category as 0, 1, 2, ...)."""
    text = column.map(as_text)
    ordered = category_order(text)
    codes = text.map({name: i for i, name in enumerate(ordered)}).to_numpy(int)
    return ordered, codes


def dummies(codes, k):
    """One 0/1 column per category: 1 where the case falls in that category."""
    return np.eye(k)[codes]

In [ ]:
%%writefile collinear_columns.py
"""Which explanatory variables are exact linear combinations of the ones before them.

Such a variable adds no information and makes the model impossible to estimate. Each
column is centred and scaled to length 1, then regressed on the columns already kept;
if nothing is left over (squared residual length at most 1e-10) the column is
redundant. A constant column is redundant too. The *later* member of a redundant pair
is the one reported, so a variable listed first survives.
"""

import numpy as np


def collinear_columns(values, names, tolerance=1e-10):
    centred = values - values.mean(axis=0)
    scale = np.linalg.norm(centred, axis=0)
    scale[scale == 0] = 1.0
    centred = centred / scale
    dropped, kept = [], []
    for j, name in enumerate(names):
        column = centred[:, j]
        if kept:
            basis = centred[:, kept]
            column = column - basis @ np.linalg.lstsq(basis, column, rcond=None)[0]
        if column @ column <= tolerance or values[:, j].max() == values[:, j].min():
            dropped.append(name)
        else:
            kept.append(j)
    return dropped

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from categories import as_text, category_order
from collinear_columns import collinear_columns

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# Text in an explanatory column counts as missing.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Preferred payment"
predictors = ["Age"]
frame = data[predictors].apply(pd.to_numeric, errors="coerce")
max_iter = 50
confidence = 95 / 100

## Cases

In [ ]:
y_ok = data[response].notna()
x_ok = frame.notna().all(axis=1)
summary = pd.DataFrame(
    {"n": [(y_ok & x_ok).sum(), (~y_ok & x_ok).sum(), (y_ok & ~x_ok).sum(),
           (~y_ok & ~x_ok).sum(), len(frame)]},
    index=["valid", "response missing", "explanatory missing", "both missing", "total"],
)
summary["%"] = summary["n"] / len(frame)
print(summary)
valid = frame[y_ok & x_ok]
# A category as text: a number read as 2.0 is "2".
outcome = data.loc[y_ok & x_ok, response].map(as_text).to_numpy()
n = len(outcome)

# Categories in order: from the smallest up when every one is a number (1, 2, 3, 10),
# otherwise as a Japanese Excel sorts text (by Shift-JIS bytes; categories.py). The last
# one is the baseline: every coefficient compares a category with it.
levels = category_order(outcome)
codes = np.array([levels.index(v) for v in outcome])
counts = np.bincount(codes, minlength=len(levels))
response_summary = pd.DataFrame({"n": [*counts, n]}, index=[*levels, "total"])
response_summary["%"] = response_summary["n"] / n
print()
print(response_summary)

## Descriptive statistics and correlations, overall and by category

In [ ]:
groups = {"all": valid, **{level: valid[codes == k] for k, level in enumerate(levels)}}
descriptive = pd.concat({
    name: pd.DataFrame({"n": len(part), "mean": part.mean(), "variance": part.var(ddof=1),
                        "SD": part.std(ddof=1), "min": part.min(), "max": part.max()})
    for name, part in groups.items()
})
correlation = pd.concat({name: part.corr() for name, part in groups.items()})
print()
print(descriptive)
print()
print(correlation)

## Variables that are linear combinations of others

In [ ]:
values = valid.to_numpy(float)
dropped = collinear_columns(values, predictors)
print("\nLinearly dependent:", dropped or "none")
# They are left out of the model.
predictors = [name for name in predictors if name not in dropped]
values = valid[predictors].to_numpy(float)

## The model

In [ ]:
# The constant is the last column, as the app prints it.
design = np.column_stack([values, np.ones(n)])
names = [*predictors, "Constant"]
slopes = design.shape[1] - 1
# statsmodels' MNLogit takes code 0 as its baseline, so the last category is renumbered
# 0 for the fit. Newton-Raphson, as in the app.
fit_codes = np.where(codes == len(levels) - 1, 0, codes + 1)


# -2 log likelihood (and the fit) of the model on these design columns.
def deviance_of(columns):
    result = sm.MNLogit(fit_codes, columns).fit(disp=0, maxiter=max_iter, method="newton")
    return -2 * result.llf, result


deviance, model = deviance_of(design)
# With only a constant, each category's probability is its share of the cases.
null_deviance = -2 * (counts * np.log(counts / n)).sum()

# Pseudo R²: Cox-Snell, Nagelkerke and McFadden.
cox_snell = 1 - np.exp(-(null_deviance - deviance) / n)
accuracy = pd.DataFrame([[cox_snell, cox_snell / (1 - np.exp(-null_deviance / n)),
                          1 - deviance / null_deviance]],
                        columns=["Cox-Snell", "Nagelkerke", "McFadden"])
print()
print(accuracy.to_string(index=False))

contrasts = len(levels) - 1
df = slopes * contrasts
chi2 = null_deviance - deviance
model_tests = pd.DataFrame([[null_deviance, np.nan, np.nan, np.nan],
                            [deviance, chi2, df, stats.chi2.sf(chi2, df)]],
                           index=["without the variables", "with the variables"],
                           columns=["-2LL", "chi-square", "df", "P"])
print()
print(model_tests)

# Each term's likelihood-ratio test: refit without it, on (categories - 1) df.
rows = []
for j in range(design.shape[1]):
    others = np.delete(design, j, axis=1)
    reduced = deviance_of(others)[0] if others.shape[1] else null_deviance
    rows.append([reduced, deviance, reduced - deviance, contrasts,
                 stats.chi2.sf(reduced - deviance, contrasts)])
variable_tests = pd.DataFrame(rows, index=names,
                              columns=["reduced -2LL", "final -2LL", "chi-square", "df", "P"])
print()
print(variable_tests)

## Coefficients: each category against the baseline

In [ ]:
# Column c of statsmodels' estimates is category c (the fit's code c + 1).
z = stats.norm.ppf(0.5 + confidence / 2)
blocks = []
for c, level in enumerate(levels[:-1]):
    b, se = np.asarray(model.params)[:, c], np.asarray(model.bse)[:, c]
    blocks.append(pd.DataFrame({
        "category": level, "b": b, "SE": se, "lower": b - z * se, "upper": b + z * se,
        "odds ratio": np.exp(b), "OR lower": np.exp(b - z * se), "OR upper": np.exp(b + z * se),
        "Wald": (b / se) ** 2, "df": 1, "P": stats.chi2.sf((b / se) ** 2, 1),
    }, index=names))
coefficients = pd.concat(blocks)
print()
print(coefficients)

## Classification table

In [ ]:
# Probabilities back in display order (the fit's code 0 is the last category); each case
# is assigned to its most probable category.
fitted = model.predict()
probabilities = np.column_stack([fitted[:, 1:], fitted[:, 0]])
predicted = probabilities.argmax(axis=1)
classification = pd.crosstab(codes, predicted).reindex(
    index=range(len(levels)), columns=range(len(levels)), fill_value=0)
classification.index = classification.columns = levels
classification["hit rate"] = np.diag(classification[levels]) / classification[levels].sum(axis=1)
hit_rate = (predicted == codes).mean()
print()
print(classification)
print(f"overall hit rate: {hit_rate:.4f}")

## Fitted probabilities

In [ ]:
predictions = pd.DataFrame(probabilities, columns=levels)
predictions.insert(0, "predicted", [levels[k] for k in predicted])
predictions.insert(0, "observed", outcome)
predictions.insert(0, "no", np.arange(1, n + 1))
print()
print(predictions.to_string(index=False))

## What the result says

Multinomial logistic regression models a choice among more than two options. Here the mobile wallet is the reference option, and the model compares card and cash with it.

Age matters: the likelihood ratio test gives a chi-square of 16.5 and P = 0.0003, a result that chance alone would rarely give. The odds ratio says how the odds multiply for each extra year of age. For card against wallet it is 1.08 (P = 0.008), and for cash against wallet 1.13 (P = 0.0008). So older customers lean towards card and cash, and away from the phone wallet. Cash is the choice of 12 of the 59 customers, whose mean age is 51.5.

But the classification table puts only 51% of customers in the right group, and only 1 of the 12 cash users. Age shifts the pattern, but age alone cannot tell you how any one customer will pay.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.